### i need to get date range for API parameters, so for that i am going to make a function that will deal with it


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests

import pandas as pd

import requests_cache
from retry_requests import retry

end_date = dt.now().strftime("%Y-%m-%d")
end_date

In [ ]:
start_date = dt.now() - td(days=7)
start_date = start_date.strftime("%y-%m-%d")
start_date

### this reminds me of the operator overloading i learned, notice we are subtracting class from class object.

### its working because in module we can define `__sub__` and control its behavior which allows it to handle such things


In [ ]:
def parameter_builder(file_path):
    df = pd.read_csv(file_path)

    # calcualting date based on current date
    end_date = dt.now().strftime("%Y-%m-%d")
    diff = dt.now() - td(days=7)
    start_date = diff.strftime("%Y-%m-%d")

    for _, row in df.iterrows():
        params = {
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date,
        }

        yield row["site_code"], params

In [ ]:
# lets see if it works as intended
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(i, param)

### ok now i am gonna need a function that will fetch the data


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry
from sqlalchemy import engine

import openmeteo_requests
import pandas as pd
import requests_cache

# retries and backoff factors can handle errors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(site_code, params):
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    print(type(responses))
    print(response)

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

### Testing the output we get


In [ ]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    fetch_weather_data(site_code=param[0], params=param[1])

In [ ]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(param)

### My project Bottle Necks:

1. I am sending 10,000 requests one at a time which is stupid instead i can create a batch of 1000 sites which is a limit and get 1000 sites data in one request.

2. I am also writing the data in database frequently which is also stupid.

3. I should use `itertuple()` instead of `iterrows()`


### Why use intertuple() instead of interrows()?

iterrows() create pandas series which consumes time instead intertuple is like python generator, it creates NameTuples like:

```
Pandas(
    index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```

in other words they are like:

```
yield(
   index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```


### why DataFrame() is faster?

- its because DataFrame() in pandas create each colum into numpy.array() which is very beneficial as numpy uses C language for execution


### I am going to try to get all the data in batches of 100 sites per request


## Method 1:


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry

import openmeteo_requests
import pandas as pd
import requests_cache
import database as db
import time

# cache_session, retries and backoff factors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=2, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"


# just a class for raising custom built error
class APIRateLimitError(Exception):
    "Raised when the Open-Meteo hourly rate limit exceeded"

    pass


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def batch_builder(file_path):
    df = pd.read_csv(file_path)

    # batch_size for each request 100 is the limit
    batch_size = 100

    for start_index in range(0, len(df), batch_size):

        # now i have a chunk of dataframe that i can work with
        df_batch = df.iloc[start_index : start_index + batch_size]

        yield df_batch


def data_parser(site_code, response):
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    return hourly_dataframe


def fetch_weather_data(df_batch):
    attempts = 3

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": df_batch["latitude"].tolist(),
        "longitude": df_batch["longitude"].tolist(),
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "shortwave_radiation",
        ],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }
    for attempt in range(attempts):
        try:
            responses = openmeteo.weather_api(url=URL, params=params)
            break

        except Exception as e:
            error_message = str(e)
            # print(f"{site_code} failed")
            print(f"Attempt {attempt + 1} / {attempts}")
            print(f"reason of faliure {e}")

            # raise error if hourly limit reached
            if "Hourly API request limit exceeded" in error_message:
                raise APIRateLimitError("API hourly limit reached")

            # if request failed then retry that request after 5 sec
            if attempt < attempts - 1:
                print("retrying after 5 seconds \n")
                time.sleep(5)

            else:
                print(f"request still failed after {attempts} attempts")
                print(f"reason of faliure: {e}")
                return None

        weather_data_batch = []
        for site_code, response in zip(df_batch["site_code"], responses):
            weather_data_batch.append(data_parser(site_code, response))

        return pd.concat(weather_data_batch, ignore_index=True)


def run_weather_etl(path):

    insert_meta_data(path)
    count = 0

    try:
        for df_batch in batch_builder(path):
            count += 1
            df = fetch_weather_data(df_batch)

            if df is not None:
                insert_weather_data(df)
                print(count)

    except APIRateLimitError as e:
        print(e)
        print("stopping ETL because rate limit reached")

### Writing the code again with a different approach

- we are gona use .apply() instead of generator and for loop see if it makes it faster
- The response object from openmeteo_requests isn't a plain Python dict/JSON — it's a special object from their SDK (backed by FlatBuffers).


### How requests_cache + SQLite works:

- It stores a mapping: cache key → full HTTP response (headers, body, status code — everything).
- The cache key is built from the request itself: method + URL + all query parameters (your params dict), hashed together.
- When you make a request, it checks: "have I seen this exact URL+params combination before, within expire_after seconds?" If yes → returns the stored response, no network call. If no → sends the real request, then stores the new response under a new key.


### When openmeteo_requests.Client.weather_api() runs, under the hood it does two things in sequence:

- Sends a plain HTTP GET request (via the session you gave it) → gets back raw bytes in the response body (FlatBuffers binary format, not JSON).
- Immediately parses those bytes into a WeatherApiResponse object using a function called WeatherApiResponse.GetRootAs() (from the openmeteo_sdk package) — this is what gives you .Latitude(), .Hourly(), etc.


### Problems with the following design:

- .apply() restarts everytime we can not create a variable that can store something and when .apply reads another row it will reset. So we have to store that variable outside the function that we will use in .apply().

- If parser() catches its own exception internally (like i am doing with try/except) and just returns normally — .apply() keeps calling parser() for the next row automatically. No crash, no restart. This part is fine.

- If an exception escapes parser() uncaught, the entire .apply() call dies immediately — and yes, that matches what my instructor said: "you'd have to rerun the whole script from row 1, and any progress not yet saved elsewhere is lost."

- .apply() either finishes completely or dies with nothing returned


## Method 2:


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry

import openmeteo_requests
import pandas as pd
import requests_cache
import database as db
import time
import sys
import csv

# retries and backoff factors
retry_session = retry(retries=2, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"
START_TIME = dt.now()
TRACKING_PATH = "tracking.csv"


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def log_status(site_code, status):
    """
    This function will append site_code and and status into a tracking csv file,
    which will keep the track of which site's data extraction successed and which failed
    """
    with open(TRACKING_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([site_code, status])


def parser(row):
    site_code = row["site_code"]

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }

    try:
        # sending request to API
        responses = openmeteo.weather_api(url=URL, params=params)

    except Exception as e:
        error_message = str(e)

        # raise error if hourly limit reached
        if "Hourly API request limit exceeded" in error_message:
            time_elapsed = dt.now() - START_TIME
            remaining_time = td(hours=1) - time_elapsed
            wait_seconds = max(
                remaining_time.total_seconds(), 0
            )  # in case if Hourly limit error hit unexpectidly after an hour for some reason
            time.sleep(wait_seconds)
            print(f"Hourly API request limit exceeded: {wait_seconds}")

        elif "Minutely API request limit exceeded" in error_message:
            time_elapsed = dt.now() - START_TIME
            remaining_time = td(minutes=1) - time_elapsed
            wait_seconds = max(
                remaining_time.total_seconds(), 0
            )  # same reason as above
            time.sleep(wait_seconds)
            print(f"Minutely API request limit exceeded, waiting for: {wait_seconds}")

        elif "Daily API request limit exceeded" in error_message:
            sys.exit("Daily API request limit reached, give it a rest see ya tomorrow")

        # as all conditions are exausted we need to update the status as a faliure
        # and i am returning None so i can handle return values in fetch_weather_data()
        log_status(site_code=site_code, status=False)
        return None

    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    # response did not failed so status is going to be True
    log_status(site_code=site_code, status=True)
    return hourly_dataframe


def new_sites_fetch_data(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    global START_TIME
    START_TIME = dt.now()

    # inserting the meta data in database
    insert_meta_data(path)

    # refreshing the tracking.csv first
    with open("tracking.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["site_code", "status"])

    # fetching the data
    df = pd.read_csv(path)
    df_series = df.apply(parser, axis=1)

    # filtered the data as failed values returned None
    df_filtered = [data for data in df_series if data is not None]

    # a small check in case if the list is empty
    if not df_filtered:
        print("There is nothing to insert into database, Please try again!")
        return

    result_df = pd.concat(df_filtered)

    # inserting the data in database
    insert_weather_data(result_df)


def failed_sites_retry(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    global START_TIME
    START_TIME = dt.now()

    # using tracker to filter sites that failed
    df = pd.read_csv(path)
    tracker = pd.read_csv("tracking.csv")

    # filtering only sites that failed
    failed_sites = tracker.loc[tracker["status"] == False]
    mask = df["site_code"].isin(failed_sites["site_code"])
    df_failed_sites = df.loc[mask]

    # now using dataframe of failed sites only we are going to retry them
    df_series = df_failed_sites.apply(parser, axis=1)
    df_filtered = [data for data in df_series if data is not None]

    # a small check in case if the list is empty
    if not df_filtered:
        print("There is nothing to insert into database, Please try again!")
        return
    result_df = pd.concat(df_filtered)

    # now we are going to insert the filtered data in database
    insert_weather_data(result_df)

## Improvements:

- Any site must not fail when we try for failed sites, we are gonna use recurrsion or while loop for that.


In [2]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests
import pandas as pd
import database as db
import time
import sys

openmeteo = openmeteo_requests.Client()

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def create_tracker_table():
    db.create_tracking_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def parser(row):
    site_code = row["site_code"]

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }

    while True:
        try:
            responses = openmeteo.weather_api(url=URL, params=params)
            break

        except Exception as e:
            error_message = str(e)

            print("EXCEPTION TYPE:", type(e))
            print("EXCEPTION:", repr(e))

            if "Hourly API request limit exceeded" in error_message:
                print("Hourly API request limit exceeded. Waiting 1 hour...")
                time.sleep(3600)
                continue

            elif "Minutely API request limit exceeded" in error_message:
                print("Minutely API request limit exceeded. Waiting 1 minute...")
                time.sleep(60)
                continue

            elif "Daily API request limit exceeded" in error_message:
                sys.exit(
                    "Daily API request limit reached, give it a rest see ya tomorrow"
                )

            db.update_tracking_status(site_code=site_code, status=False)
            return None

    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    # response did not failed so status is going to be True
    db.update_tracking_status(site_code=site_code, status=True)
    return hourly_dataframe


def new_sites_fetch_data(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.

    # batch size
    batch_size = 100

    # inserting the meta data in database
    insert_meta_data(path)

    # fetching the data and filtering the sites that are already done
    df = pd.read_csv(path)
    tracker = db.read_parsed_sites()

    mask = ~df["site_code"].isin(tracker["site_code"])
    df = df.loc[mask]

    # instead of getting the data of 10,000 sites we are gonna insert 100 sites data over time
    for start in range(0, len(df), batch_size):

        batch_df = df.iloc[start : start + batch_size]
        df_series = batch_df.apply(parser, axis=1)

        # filtered the data as failed values returned None
        parsed_data = [data for data in df_series if data is not None]

        # a small check in case if the list is empty
        if not parsed_data:
            print("There is nothing to insert into database, Please try again!")
            return

        result_df = pd.concat(parsed_data)

        # inserting the data in database
        insert_weather_data(result_df)


def failed_sites_retry(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    attempts = 0
    while True:

        # using tracker to filter sites that failed
        df = pd.read_csv(path)
        failed_sites = db.read_failed_sites()

        # if there are no failed site break the loop
        if failed_sites.empty:
            break

        mask = df["site_code"].isin(failed_sites["site_code"])
        df_failed_sites = df.loc[mask]

        # now using dataframe of failed sites only we are going to retry them
        df_series = df_failed_sites.apply(parser, axis=1)

        df_filtered = [data for data in df_series if data is not None]

        if not df_filtered:
            attempts += 1
            print(f"Still no data is returned, attempt Number: {attempts}")
            continue

        result_df = pd.concat(df_filtered)

        # now we are going to insert the filtered data in database
        insert_weather_data(result_df)

In [ ]:
create_meta_table()
create_weather_data()
create_tracker_table()


new_sites_fetch_data(PATH)

In [1]:
import requests

params = {
    "latitude": 42.3601,
    "longitude": -71.0589,
    "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
    "timezone": "auto",
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)

print(response.status_code)
print(response.json())

200
{'latitude': 42.365166, 'longitude': -71.0618, 'generationtime_ms': 0.12135505676269531, 'utc_offset_seconds': -14400, 'timezone': 'America/New_York', 'timezone_abbreviation': 'GMT-4', 'elevation': 9.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'shortwave_radiation': 'W/m²'}, 'hourly': {'time': ['2026-09-19T00:00', '2026-09-19T01:00', '2026-09-19T02:00', '2026-09-19T03:00', '2026-09-19T04:00', '2026-09-19T05:00', '2026-09-19T06:00', '2026-09-19T07:00', '2026-09-19T08:00', '2026-09-19T09:00', '2026-09-19T10:00', '2026-09-19T11:00', '2026-09-19T12:00', '2026-09-19T13:00', '2026-09-19T14:00', '2026-09-19T15:00', '2026-09-19T16:00', '2026-09-19T17:00', '2026-09-19T18:00', '2026-09-19T19:00', '2026-09-19T20:00', '2026-09-19T21:00', '2026-09-19T22:00', '2026-09-19T23:00', '2026-09-20T00:00', '2026-09-20T01:00', '2026-09-20T02:00', '2026-09-20T03:00', '2026-09-20T04:00', '2026-09-20T05:00', '2026-09-20T06:00', '2026-09-20T07:00', '2026-09-20T

In [ ]:
{
    "latitude": 42.365166,
    "longitude": -71.0618,
    "generationtime_ms": 0.12135505676269531,
    "utc_offset_seconds": -14400,
    "timezone": "America/New_York",
    "timezone_abbreviation": "GMT-4",
    "elevation": 9.0,
    "hourly_units": {
        "time": "iso8601",
        "temperature_2m": "°C",
        "relative_humidity_2m": "%",
        "shortwave_radiation": "W/m²",
    },
    "hourly": {
        "time": [
            "2026-09-19T00:00",
            "2026-09-19T01:00",
            "2026-09-19T02:00",
            "2026-09-19T03:00",
            "2026-09-19T04:00",
            "2026-09-19T05:00",
            "2026-09-19T06:00",
            "2026-09-19T07:00",
            "2026-09-19T08:00",
            "2026-09-19T09:00",
            "2026-09-19T10:00",
            "2026-09-19T11:00",
            "2026-09-19T12:00",
            "2026-09-19T13:00",
            "2026-09-19T14:00",
            "2026-09-19T15:00",
            "2026-09-19T16:00",
            "2026-09-19T17:00",
            "2026-09-19T18:00",
            "2026-09-19T19:00",
            "2026-09-19T20:00",
            "2026-09-19T21:00",
            "2026-09-19T22:00",
            "2026-09-19T23:00",
            "2026-09-20T00:00",
            "2026-09-20T01:00",
            "2026-09-20T02:00",
            "2026-09-20T03:00",
            "2026-09-20T04:00",
            "2026-09-20T05:00",
            "2026-09-20T06:00",
            "2026-09-20T07:00",
            "2026-09-20T08:00",
            "2026-09-20T09:00",
            "2026-09-20T10:00",
            "2026-09-20T11:00",
            "2026-09-20T12:00",
            "2026-09-20T13:00",
            "2026-09-20T14:00",
            "2026-09-20T15:00",
            "2026-09-20T16:00",
            "2026-09-20T17:00",
            "2026-09-20T18:00",
            "2026-09-20T19:00",
            "2026-09-20T20:00",
            "2026-09-20T21:00",
            "2026-09-20T22:00",
            "2026-09-20T23:00",
            "2026-09-21T00:00",
            "2026-09-21T01:00",
            "2026-09-21T02:00",
            "2026-09-21T03:00",
            "2026-09-21T04:00",
            "2026-09-21T05:00",
            "2026-09-21T06:00",
            "2026-09-21T07:00",
            "2026-09-21T08:00",
            "2026-09-21T09:00",
            "2026-09-21T10:00",
            "2026-09-21T11:00",
            "2026-09-21T12:00",
            "2026-09-21T13:00",
            "2026-09-21T14:00",
            "2026-09-21T15:00",
            "2026-09-21T16:00",
            "2026-09-21T17:00",
            "2026-09-21T18:00",
            "2026-09-21T19:00",
            "2026-09-21T20:00",
            "2026-09-21T21:00",
            "2026-09-21T22:00",
            "2026-09-21T23:00",
            "2026-09-22T00:00",
            "2026-09-22T01:00",
            "2026-09-22T02:00",
            "2026-09-22T03:00",
            "2026-09-22T04:00",
            "2026-09-22T05:00",
            "2026-09-22T06:00",
            "2026-09-22T07:00",
            "2026-09-22T08:00",
            "2026-09-22T09:00",
            "2026-09-22T10:00",
            "2026-09-22T11:00",
            "2026-09-22T12:00",
            "2026-09-22T13:00",
            "2026-09-22T14:00",
            "2026-09-22T15:00",
            "2026-09-22T16:00",
            "2026-09-22T17:00",
            "2026-09-22T18:00",
            "2026-09-22T19:00",
            "2026-09-22T20:00",
            "2026-09-22T21:00",
            "2026-09-22T22:00",
            "2026-09-22T23:00",
            "2026-09-23T00:00",
            "2026-09-23T01:00",
            "2026-09-23T02:00",
            "2026-09-23T03:00",
            "2026-09-23T04:00",
            "2026-09-23T05:00",
            "2026-09-23T06:00",
            "2026-09-23T07:00",
            "2026-09-23T08:00",
            "2026-09-23T09:00",
            "2026-09-23T10:00",
            "2026-09-23T11:00",
            "2026-09-23T12:00",
            "2026-09-23T13:00",
            "2026-09-23T14:00",
            "2026-09-23T15:00",
            "2026-09-23T16:00",
            "2026-09-23T17:00",
            "2026-09-23T18:00",
            "2026-09-23T19:00",
            "2026-09-23T20:00",
            "2026-09-23T21:00",
            "2026-09-23T22:00",
            "2026-09-23T23:00",
            "2026-09-24T00:00",
            "2026-09-24T01:00",
            "2026-09-24T02:00",
            "2026-09-24T03:00",
            "2026-09-24T04:00",
            "2026-09-24T05:00",
            "2026-09-24T06:00",
            "2026-09-24T07:00",
            "2026-09-24T08:00",
            "2026-09-24T09:00",
            "2026-09-24T10:00",
            "2026-09-24T11:00",
            "2026-09-24T12:00",
            "2026-09-24T13:00",
            "2026-09-24T14:00",
            "2026-09-24T15:00",
            "2026-09-24T16:00",
            "2026-09-24T17:00",
            "2026-09-24T18:00",
            "2026-09-24T19:00",
            "2026-09-24T20:00",
            "2026-09-24T21:00",
            "2026-09-24T22:00",
            "2026-09-24T23:00",
            "2026-09-25T00:00",
            "2026-09-25T01:00",
            "2026-09-25T02:00",
            "2026-09-25T03:00",
            "2026-09-25T04:00",
            "2026-09-25T05:00",
            "2026-09-25T06:00",
            "2026-09-25T07:00",
            "2026-09-25T08:00",
            "2026-09-25T09:00",
            "2026-09-25T10:00",
            "2026-09-25T11:00",
            "2026-09-25T12:00",
            "2026-09-25T13:00",
            "2026-09-25T14:00",
            "2026-09-25T15:00",
            "2026-09-25T16:00",
            "2026-09-25T17:00",
            "2026-09-25T18:00",
            "2026-09-25T19:00",
            "2026-09-25T20:00",
            "2026-09-25T21:00",
            "2026-09-25T22:00",
            "2026-09-25T23:00",
        ],
        "temperature_2m": [
            15.8,
            14.6,
            14.0,
            13.7,
            12.6,
            11.7,
            11.0,
            10.4,
            12.3,
            14.4,
            16.0,
            16.7,
            16.9,
            17.1,
            17.1,
            17.1,
            16.6,
            16.2,
            15.6,
            14.5,
            13.6,
            13.2,
            12.4,
            11.7,
            11.0,
            10.6,
            10.3,
            9.8,
            9.6,
            9.4,
            9.2,
            9.3,
            12.7,
            15.6,
            16.8,
            17.1,
            16.7,
            16.8,
            16.6,
            15.5,
            14.7,
            14.4,
            14.5,
            14.5,
            14.7,
            15.0,
            15.2,
            15.5,
            15.6,
            15.9,
            16.0,
            14.9,
            13.8,
            13.6,
            13.5,
            13.4,
            13.3,
            13.4,
            14.0,
            14.5,
            15.4,
            15.9,
            16.5,
            16.3,
            16.3,
            15.9,
            15.6,
            15.0,
            15.1,
            15.2,
            15.3,
            15.3,
            15.0,
            14.6,
            14.4,
            14.0,
            14.0,
            14.4,
            14.4,
            14.6,
            14.9,
            14.7,
            14.5,
            14.4,
            14.9,
            14.6,
            14.5,
            14.4,
            14.2,
            13.9,
            13.5,
            13.3,
            13.6,
            13.9,
            14.0,
            14.3,
            14.3,
            14.4,
            14.3,
            14.4,
            14.3,
            14.4,
            14.5,
            14.6,
            14.5,
            14.8,
            14.6,
            15.2,
            15.4,
            15.4,
            14.7,
            14.5,
            14.3,
            14.0,
            14.0,
            14.0,
            14.2,
            14.5,
            14.8,
            14.8,
            15.1,
            14.9,
            15.1,
            14.9,
            14.7,
            14.6,
            14.5,
            14.6,
            14.7,
            14.9,
            15.1,
            15.3,
            15.2,
            15.0,
            14.8,
            14.7,
            14.6,
            14.5,
            14.2,
            13.9,
            13.8,
            14.0,
            14.4,
            14.6,
            14.2,
            13.5,
            13.0,
            12.9,
            12.9,
            12.9,
            12.5,
            12.0,
            11.6,
            11.3,
            11.2,
            11.1,
            11.1,
            11.1,
            11.1,
            10.9,
            10.6,
            10.5,
            10.6,
            10.9,
            11.2,
            11.4,
            11.6,
            11.8,
        ],
        "relative_humidity_2m": [
            55,
            59,
            59,
            61,
            65,
            70,
            70,
            74,
            63,
            62,
            53,
            50,
            51,
            50,
            48,
            48,
            50,
            50,
            51,
            55,
            60,
            64,
            69,
            72,
            73,
            75,
            70,
            73,
            77,
            77,
            77,
            79,
            79,
            65,
            59,
            56,
            56,
            54,
            60,
            75,
            80,
            83,
            84,
            89,
            93,
            93,
            93,
            93,
            94,
            96,
            97,
            96,
            96,
            95,
            95,
            94,
            92,
            90,
            87,
            83,
            73,
            66,
            61,
            62,
            62,
            63,
            65,
            67,
            67,
            67,
            67,
            66,
            66,
            66,
            65,
            67,
            71,
            71,
            71,
            70,
            71,
            68,
            66,
            65,
            61,
            60,
            60,
            59,
            60,
            61,
            62,
            60,
            57,
            56,
            58,
            58,
            59,
            59,
            60,
            59,
            60,
            62,
            62,
            63,
            66,
            63,
            60,
            58,
            59,
            60,
            62,
            62,
            63,
            68,
            70,
            73,
            73,
            72,
            68,
            67,
            63,
            67,
            65,
            65,
            66,
            67,
            67,
            68,
            68,
            67,
            67,
            67,
            70,
            73,
            76,
            77,
            77,
            77,
            79,
            82,
            83,
            79,
            72,
            68,
            71,
            76,
            80,
            79,
            76,
            75,
            78,
            83,
            87,
            89,
            89,
            90,
            91,
            92,
            92,
            92,
            91,
            91,
            91,
            92,
            92,
            92,
            91,
            91,
        ],
        "shortwave_radiation": [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            9.0,
            151.0,
            337.0,
            523.0,
            657.0,
            741.0,
            768.0,
            678.0,
            663.0,
            476.0,
            350.0,
            149.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            7.0,
            144.0,
            264.0,
            418.0,
            304.0,
            84.0,
            95.0,
            58.0,
            71.0,
            44.0,
            30.0,
            35.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            9.0,
            39.0,
            78.0,
            134.0,
            198.0,
            270.0,
            526.0,
            481.0,
            532.0,
            363.0,
            174.0,
            21.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            3.0,
            34.0,
            78.0,
            116.0,
            153.0,
            427.0,
            507.0,
            410.0,
            526.0,
            357.0,
            252.0,
            96.0,
            16.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            4.0,
            25.0,
            88.0,
            142.0,
            519.0,
            665.0,
            583.0,
            261.0,
            201.0,
            153.0,
            80.0,
            41.0,
            6.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            1.0,
            32.0,
            87.0,
            133.0,
            167.0,
            179.0,
            186.0,
            181.0,
            192.0,
            134.0,
            67.0,
            16.0,
            1.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            6.0,
            17.0,
            25.0,
            30.0,
            32.0,
            30.0,
            25.0,
            17.0,
            13.0,
            9.0,
            5.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ],
    },
}